In [1]:
print("kernel alive")

kernel alive


In [2]:
from experiments.robot.libero.gen_config import GenerateConfig
from experiments.robot.openvla_utils import get_action_head, get_processor, get_proprio_projector, get_vla
from prismatic.vla.constants import NUM_ACTIONS_CHUNK, PROPRIO_DIM


# Instantiate config (see class GenerateConfig in experiments/robot/libero/run_libero_eval.py for definitions)
cfg = GenerateConfig(
    pretrained_checkpoint = "openvla_7b_oft_finetuned",
    use_l1_regression = True,
    use_diffusion = False,
    use_film = False,
    num_images_in_input = 2,
    use_proprio = True,
    load_in_8bit = False,
    load_in_4bit = True,
    center_crop = True,
    num_open_loop_steps = NUM_ACTIONS_CHUNK,
    unnorm_key = "libero_spatial_no_noops",
)

# Load OpenVLA-OFT policy and inputs processor
vla  = get_vla(cfg)
processor = get_processor(cfg)

# Load MLP action head to generate continuous actions (via L1 regression)
action_head = get_action_head(cfg, llm_dim=vla.llm_dim)

# Load proprio projector to map proprio to language embedding space
proprio_projector = get_proprio_projector(cfg, llm_dim=vla.llm_dim, proprio_dim=PROPRIO_DIM)

2026-06-29 18:14:33.523486: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-29 18:14:33.610300: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-06-29 18:14:33.610364: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-29 18:14:33.615519: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-06-29 18:14:33.645256: I tensorflow/core/platform/cpu_feature_guar

Using LIBERO constants:
  NUM_ACTIONS_CHUNK = 8
  ACTION_DIM = 7
  PROPRIO_DIM = 8
  ACTION_PROPRIO_NORMALIZATION_TYPE = bounds_q99
If needed, manually set the correct constants in `prismatic/vla/constants.py`!


2026-06-29 18:14:41.761944: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-06-29 18:14:41.768361: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2256] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Instantiating pretrained VLA policy...


Loading checkpoint shards: 100%|██████████| 4/4 [00:17<00:00,  4.50s/it]


In [3]:
import pickle
from experiments.robot.openvla_utils import get_vla_action

with open("experiments/robot/libero/sample_libero_spatial_observation.pkl", "rb") as file:
    observation = pickle.load(file)

# Generate robot action chunk (sequence of future actions)
actions = get_vla_action(cfg, vla, processor, observation, observation["task_description"], action_head, proprio_projector)
print("Generated action chunk:")
for act in actions:
    print(act)

using modeling_prismatic.py in openvla-7b-oft-fintuned... folder
this is where ReVIP should happen
Generated action chunk:
[0.153 0.034 0.050 -0.004 0.037 0.006 1.008]
[0.246 0.078 0.104 -0.010 0.039 0.005 1.008]
[0.356 0.166 0.136 -0.009 0.044 0.002 1.008]
[0.515 0.254 0.162 -0.003 0.052 0.003 1.008]
[0.665 0.325 0.167 -0.002 0.051 0.006 1.008]
[0.799 0.404 0.148 -0.003 0.026 0.005 1.008]
[0.842 0.455 0.106 -0.003 0.006 0.007 1.008]
[0.842 0.486 0.072 -0.002 0.002 0.013 1.000]


### Batch mask attention Genneration

In [4]:
import torch
import numpy as np

IGNORE_INDEX = -100
ACTION_TOKEN_ID = 29871

In [5]:
def make_prompts(task_labels):
    return [
        f"In: What action should the robot take to {task.lower()}?\nOut:"
        for task in task_labels
    ]

def make_text_batch(task_labels, processor, device):
    prompts = make_prompts(task_labels)

    encoded = [
        processor.tokenizer(prompt, return_tensors="pt")
        for prompt in prompts
    ]

    input_ids = torch.cat([x["input_ids"] for x in encoded], dim=0).to(device)
    attention_mask = torch.cat([x["attention_mask"] for x in encoded], dim=0).to(device)

    return input_ids.long(), attention_mask.long(), prompts

In [6]:
from experiments.robot.openvla_utils import normalize_proprio

def make_proprio_batch(obs_batch, cfg, vla, device, dtype):
    """Normalize and stack proprio states as [B, PROPRIO_DIM].

    This mirrors get_vla_action(): raw obs["state"] is normalized with
    vla.norm_stats[cfg.unnorm_key]["proprio"] before it is sent through the
    proprio projector.
    """
    if not getattr(cfg, "use_proprio", False):
        return None

    proprio_norm_stats = vla.norm_stats[cfg.unnorm_key]["proprio"]
    proprio_list = []

    for i, obs in enumerate(obs_batch):
        if "state" not in obs:
            raise KeyError(
                f"obs_batch[{i}] has no 'state'. The LIBERO loader must put raw proprio in obs['state']."
            )

        proprio = normalize_proprio(np.asarray(obs["state"]), proprio_norm_stats)
        proprio_list.append(torch.as_tensor(proprio, dtype=torch.float32))

    return torch.stack(proprio_list, dim=0).to(device=device, dtype=dtype)

In [7]:
from experiments.robot.openvla_utils import prepare_images_for_vla

def make_pixel_batch(obs_batch, prompts, cfg, processor, device, dtype):
    pixel_values = []

    for obs, prompt in zip(obs_batch, prompts):
        images = [obs["full_image"]]

        if cfg.num_images_in_input > 1:
            images += [v for k, v in obs.items() if "wrist" in k]

        images = prepare_images_for_vla(images, cfg)

        inputs = processor(prompt, images[0]).to(device, dtype=dtype)

        if len(images) > 1:
            wrist_pixels = [
                processor(prompt, img).to(device, dtype=dtype)["pixel_values"]
                for img in images[1:]
            ]
            inputs["pixel_values"] = torch.cat(
                [inputs["pixel_values"]] + wrist_pixels,
                dim=1,
            )

        pixel_values.append(inputs["pixel_values"])

    return torch.cat(pixel_values, dim=0)

In [8]:
def add_action_token(input_ids, attention_mask):
    """Append OpenVLA's empty/action-start token when the tokenizer did not add it."""
    if not torch.all(input_ids[:, -1] == ACTION_TOKEN_ID):
        extra_ids = torch.full(
            (input_ids.shape[0], 1),
            ACTION_TOKEN_ID,
            device=input_ids.device,
            dtype=input_ids.dtype,
        )
        extra_mask = torch.ones(
            (attention_mask.shape[0], 1),
            device=attention_mask.device,
            dtype=attention_mask.dtype,
        )
        input_ids = torch.cat([input_ids, extra_ids], dim=1)
        attention_mask = torch.cat([attention_mask, extra_mask], dim=1)

    return input_ids, attention_mask


def make_action_inputs(input_ids, attention_mask, vla):
    """Prepare prompt + placeholder action tokens exactly like predict_action().

    Important: labels must be created BEFORE _prepare_input_for_action_prediction()
    so _prepare_labels_for_action_prediction() can mark the newly appended action
    placeholders. Creating labels after placeholder insertion leaves all action
    masks empty.
    """
    input_ids, attention_mask = add_action_token(input_ids, attention_mask)

    labels = torch.full_like(input_ids, IGNORE_INDEX)
    num_prompt_tokens = input_ids.shape[-1] - 1

    input_ids, attention_mask = vla._prepare_input_for_action_prediction(
        input_ids,
        attention_mask,
    )
    labels = vla._prepare_labels_for_action_prediction(labels, input_ids)

    return input_ids, attention_mask, labels, num_prompt_tokens

In [9]:
@torch.no_grad()
def make_vla_batch(
    batch,
    cfg,
    vla,
    processor,
    proprio_projector=None,
    append_proprio=True,
    device=None,
    dtype=torch.bfloat16,
):
    """Build the tensors needed by OpenVLA-OFT for a whole batch.

    If append_proprio=True, projected_patch_embeddings already contains the
    final proprio token. For ReViP/TS-FiLM, call with append_proprio=False,
    modulate vision_patch_embeddings, and append proprio afterwards.
    """
    device = device or next(vla.parameters()).device

    input_ids, attention_mask, prompts = make_text_batch(
        batch["task_labels"], processor, device
    )

    pixel_values = make_pixel_batch(
        batch["obs_batch"], prompts, cfg, processor, device, dtype
    )

    proprio = make_proprio_batch(
        batch["obs_batch"], cfg, vla, device, dtype
    )

    input_ids, attention_mask, labels, num_prompt_tokens = make_action_inputs(
        input_ids, attention_mask, vla
    )

    all_actions_mask = vla._process_action_masks(labels)
    input_embeddings = vla.get_input_embeddings()(input_ids)

    language_embeddings = input_embeddings[~all_actions_mask].reshape(
        input_embeddings.shape[0],
        -1,
        input_embeddings.shape[-1],
    )

    vision_patch_embeddings = vla._process_vision_features(
        pixel_values,
        language_embeddings,
        getattr(cfg, "use_film", False),
    )

    projected_patch_embeddings = vision_patch_embeddings
    use_proprio = False

    if append_proprio and proprio_projector is not None and proprio is not None:
        projected_patch_embeddings = vla._process_proprio_features(
            projected_patch_embeddings,
            proprio,
            proprio_projector,
        )
        use_proprio = True

    return {
        "vision_patch_embeddings": vision_patch_embeddings,
        "projected_patch_embeddings": projected_patch_embeddings,
        "proprio": proprio,
        "use_proprio": use_proprio,
        "num_patches": projected_patch_embeddings.shape[1],
        "num_prompt_tokens": num_prompt_tokens,
        "input_embeddings": input_embeddings,
        "all_actions_mask": all_actions_mask,
        "attention_mask": attention_mask,
        "labels": labels,
        "input_ids": input_ids,
        "pixel_values": pixel_values,
    }

### TSO loading functions

In [10]:
import re
import h5py
import torch
import numpy as np
from pathlib import Path
from torch.utils.data import Dataset, DataLoader


def task_name_from_path(path):
    stem = Path(path).stem
    stem = re.sub(r"_demo$", "", stem)
    stem = re.sub(r"^(?:[A-Z0-9]+_)+(?=[a-z])", "", stem)
    return stem.replace("_", " ")


def demo_sort_key(demo_id):
    m = re.search(r"\d+$", demo_id)
    return int(m.group()) if m else demo_id


def load_tso_by_demo(tso_dir):
    tso_dir = Path(tso_dir)
    out = {}

    for p in sorted(tso_dir.glob("*.pt")):
        data = torch.load(p, map_location="cpu")

        demo_id = data.get("source_demo_name", p.stem)
        features = data["features"]
        frame_indices = data.get("frame_indices", None)

        if frame_indices is None:
            out[demo_id] = features
        else:
            out[demo_id] = {
                int(frame_idx): feat
                for frame_idx, feat in zip(frame_indices, features)
            }

    return out


def _read_step_dataset(group, key, step):
    if group is not None and key in group:
        return np.asarray(group[key][step]).reshape(-1)
    return None


def _first_matching_dim(candidates, expected_dim):
    if expected_dim is None:
        return candidates[0] if candidates else None

    for x in candidates:
        if x is not None and x.shape[-1] == expected_dim:
            return x

    return None


def extract_proprio_state(demo_group, obs_group, step, expected_dim=None):
    """Extract raw proprio for OpenVLA-OFT.

    The returned vector becomes obs["state"].

    Later, make_proprio_batch() will normalize obs["state"] using
    vla.norm_stats[cfg.unnorm_key]["proprio"], then append it through
    vla._process_proprio_features().
    """
    candidates = []

    # Preferred explicit state fields.
    for group, key in (
        (obs_group, "state"),
        (obs_group, "states"),
        (demo_group, "state"),
        (demo_group, "states"),
    ):
        x = _read_step_dataset(group, key, step)
        if x is not None:
            candidates.append(x)

    # Common LIBERO/OpenVLA regenerated dataset layout.
    #
    # In regenerated LIBERO data, the OpenVLA-OFT proprio state is usually:
    #   ee_states + gripper_states
    #
    # For LIBERO, this should match PROPRIO_DIM, typically 8.
    key_groups = [
        ("ee_states", "gripper_states"),
        ("ee_pos", "ee_ori", "gripper_states"),
        ("robot0_eef_pos", "robot0_eef_quat", "robot0_gripper_qpos"),
        ("robot0_eef_pos", "robot0_eef_quat", "robot0_gripper_qpos", "robot0_gripper_qvel"),
        ("eef_pos", "eef_quat", "gripper_qpos"),
        ("ee_pos", "ee_quat", "gripper_qpos"),
    ]

    for keys in key_groups:
        if obs_group is not None and all(k in obs_group for k in keys):
            candidate = np.concatenate(
                [np.asarray(obs_group[k][step]).reshape(-1) for k in keys],
                axis=0,
            )
            candidates.append(candidate)

    state = _first_matching_dim(candidates, expected_dim)

    if state is None:
        available = list(obs_group.keys()) if obs_group is not None else []
        raise KeyError(
            "Could not find a proprio state with the expected dimension. "
            f"expected_dim={expected_dim}, available obs keys={available}"
        )

    return state.astype(np.float32)


def load_libero_frames(path, expected_proprio_dim=None):
    path = Path(path)
    task = task_name_from_path(path)
    samples = []

    with h5py.File(path, "r") as f:
        for demo_id in sorted(f["data"].keys(), key=demo_sort_key):
            demo = f["data"][demo_id]
            obs = demo["obs"]

            agentview = obs["agentview_rgb"][()]
            wrist = obs["eye_in_hand_rgb"][()]

            for step in range(agentview.shape[0]):
                sample = {
                    "obs": {
                        "full_image": agentview[step],
                        "wrist_image": wrist[step],

                        # Important addition:
                        # OpenVLA-OFT expects proprio as obs["state"].
                        "state": extract_proprio_state(
                            demo,
                            obs,
                            step,
                            expected_dim=expected_proprio_dim,
                        ),
                    },
                    "task_label": task,
                    "demo_id": demo_id,
                    "step": step,
                }

                samples.append(sample)

    print(f"loaded {len(samples)} frames | task: {task}")
    return samples


class FrameDataset(Dataset):
    def __init__(self, samples, tso_by_demo=None):
        self.samples = samples
        self.tso_by_demo = tso_by_demo

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        x = self.samples[i]
        out = dict(x)

        if self.tso_by_demo is not None:
            tso_demo = self.tso_by_demo[x["demo_id"]]
            out["tso"] = (
                tso_demo[x["step"]]
                if isinstance(tso_demo, dict)
                else tso_demo[x["step"]]
            )

        return out


def collate_frames(batch):
    out = {
        "obs_batch": [x["obs"] for x in batch],
        "task_labels": [x["task_label"] for x in batch],
        "demo_ids": [x["demo_id"] for x in batch],
        "steps": [x["step"] for x in batch],

        # Convenience copy of raw proprio.
        # The actual OpenVLA-OFT path still uses obs_batch[i]["state"].
        "proprio": torch.stack(
            [
                torch.as_tensor(x["obs"]["state"], dtype=torch.float32)
                for x in batch
            ]
        ),
    }

    if "tso" in batch[0]:
        out["tso"] = torch.stack([x["tso"] for x in batch])

    return out


def make_libero_dataloader(
    data_path,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    tso_dir=None,
    expected_proprio_dim=None,
):
    if expected_proprio_dim is None and "PROPRIO_DIM" in globals():
        expected_proprio_dim = PROPRIO_DIM

    samples = load_libero_frames(
        data_path,
        expected_proprio_dim=expected_proprio_dim,
    )

    tso_by_demo = load_tso_by_demo(tso_dir) if tso_dir is not None else None
    dataset = FrameDataset(samples, tso_by_demo=tso_by_demo)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        collate_fn=collate_frames,
    )

In [11]:
loader = make_libero_dataloader(
    data_path="libero_data/libero_spatial/pick_up_the_black_bowl_on_the_wooden_cabinet_and_place_it_on_the_plate_demo.hdf5",
    tso_dir="tso_data/libero_spatial/pick_up_the_black_bowl_on_the_wooden_cabinet_and_place_it_on_the_plate",
    batch_size=1,
)

loaded 6880 frames | task: pick up the black bowl on the wooden cabinet and place it on the plate


In [12]:
batch = next(iter(loader))

clue = batch["tso"].to(
    device=next(vla.parameters()).device,
    dtype=torch.bfloat16,
)

### Projected Patch Embedding Generation

In [13]:
import torch
from experiments.robot.openvla_utils import prepare_images_for_vla

IGNORE_INDEX = -100

@torch.no_grad()
def get_projected_patch_embeddings_batchwise(
    obs_batch,
    task_labels,
    cfg,
    vla,
    processor,
    device=None,
    dtype=torch.bfloat16,
):
    if device is None:
        device = next(vla.parameters()).device

    input_ids_list = []
    attention_mask_list = []
    pixel_values_list = []

    for obs, task_label in zip(obs_batch, task_labels):
        all_images = [obs["full_image"]]

        if cfg.num_images_in_input > 1:
            all_images.extend([obs[k] for k in obs.keys() if "wrist" in k])

        all_images = prepare_images_for_vla(all_images, cfg)
        primary_image = all_images.pop(0)

        prompt = f"In: What action should the robot take to {task_label.lower()}?\nOut:"

        inputs = processor(prompt, primary_image).to(device, dtype=dtype)

        if len(all_images) > 0:
            wrist_inputs = [
                processor(prompt, img).to(device, dtype=dtype)
                for img in all_images
            ]

            inputs["pixel_values"] = torch.cat(
                [inputs["pixel_values"]]
                + [x["pixel_values"] for x in wrist_inputs],
                dim=1,
            )

        input_ids_list.append(inputs["input_ids"].long())
        attention_mask_list.append(inputs["attention_mask"].long())
        pixel_values_list.append(inputs["pixel_values"])

    input_ids = torch.cat(input_ids_list, dim=0)
    attention_mask = torch.cat(attention_mask_list, dim=0)
    pixel_values = torch.cat(pixel_values_list, dim=0)

    if not torch.all(input_ids[:, -1] == 29871):
        extra_token = torch.full(
            (input_ids.shape[0], 1),
            29871,
            device=device,
            dtype=input_ids.dtype,
        )
        input_ids = torch.cat([input_ids, extra_token], dim=1)

        extra_mask = torch.ones(
            (attention_mask.shape[0], 1),
            device=device,
            dtype=attention_mask.dtype,
        )
        attention_mask = torch.cat([attention_mask, extra_mask], dim=1)

    labels = input_ids.clone()
    labels[:] = IGNORE_INDEX

    input_ids, attention_mask = vla._prepare_input_for_action_prediction(
        input_ids,
        attention_mask,
    )

    labels = vla._prepare_labels_for_action_prediction(labels, input_ids)
    all_actions_mask = vla._process_action_masks(labels)

    input_embeddings = vla.get_input_embeddings()(input_ids)

    language_embeddings = input_embeddings[~all_actions_mask].reshape(
        input_embeddings.shape[0],
        -1,
        input_embeddings.shape[2],
    )

    projected_patch_embeddings = vla._process_vision_features(
        pixel_values,
        language_embeddings,
        False,
    )

    return projected_patch_embeddings

In [14]:
patch_tokens = get_projected_patch_embeddings_batchwise(
    obs_batch=batch["obs_batch"],
    task_labels=batch["task_labels"],
    cfg=cfg,
    vla=vla,
    processor=processor,
)

### FiLM

In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ReViPTSFiLM(nn.Module):
    def __init__(
        self,
        clue_dim=2048,
        token_dim=4096,
        proj_hidden=2048,
        film_hidden=2048,
        gamma_scale=0.1,
        beta_scale=0.1,
    ):
        super().__init__()

        # Proj(.): clue -> token space
        self.clue_proj = nn.Sequential(
            nn.Linear(clue_dim, proj_hidden),
            nn.SiLU(),
            nn.Linear(proj_hidden, token_dim),
        )

        # h(.): z -> [gamma, beta]
        self.film_head = nn.Sequential(
            nn.Linear(token_dim, film_hidden),
            nn.SiLU(),
            nn.Linear(film_hidden, 2 * token_dim),
        )

        # Learnable nonnegative alpha through softplus
        # Start very small so TS-FiLM is near identity at initialization
        self.alpha_raw = nn.Parameter(torch.tensor(-0.43))

        self.gamma_scale = gamma_scale
        self.beta_scale = beta_scale

        self.reset_parameters()

    def reset_parameters(self):
        nn.init.zeros_(self.film_head[-1].bias)
        nn.init.xavier_uniform_(self.film_head[-1].weight, gain=0.01)

    def forward(self, patch_tokens, clue, return_aux=False):
        """
        patch_tokens: [B, N, D]
        clue:         [B, clue_dim]
        """
        if patch_tokens.dim() != 3:
            raise ValueError(f"patch_tokens must be [B, N, D], got {patch_tokens.shape}")
        if clue.dim() != 2:
            raise ValueError(f"clue must be [B, clue_dim], got {clue.shape}")

        B, N, D = patch_tokens.shape
        if clue.shape[0] != B:
            raise ValueError("Batch size mismatch between patch_tokens and clue")

        if D != self.film_head[-1].out_features // 2:
            raise ValueError(
                f"Token dim mismatch: patch_tokens has D={D}, "
                f"but TS-FiLM expects {self.film_head[-1].out_features // 2}"
            )

        z = self.clue_proj(clue)                    # [B, D]
        gamma_beta = self.film_head(z)             # [B, 2D]
        gamma, beta = gamma_beta.chunk(2, dim=-1)  # [B, D], [B, D]

        # Keep FiLM modulation bounded
        gamma = self.gamma_scale * torch.tanh(gamma)
        beta = self.beta_scale * torch.tanh(beta)

        alpha = F.softplus(self.alpha_raw)         # scalar >= 0

        delta = gamma[:, None, :] * patch_tokens + beta[:, None, :]
        mod_patch_tokens = patch_tokens + alpha * delta

        if return_aux:
            return mod_patch_tokens, {
                "z": z,
                "gamma": gamma,
                "beta": beta,
                "alpha": alpha,
            }

        return mod_patch_tokens
    
import torch
import torch.nn.functional as F


def revip_action_loss(
    pred_actions: torch.Tensor,
    gt_actions: torch.Tensor,
    loss_type: str = "l1",
):
    """
    Loss on generated actions vs ground-truth actions.

    Args:
        pred_actions: [B, ...]
        gt_actions:   [B, ...] same shape as pred_actions
        loss_type:    "l1", "mse", or "smooth_l1"

    Returns:
        loss: scalar tensor
        stats: dict
    """
    if pred_actions.shape != gt_actions.shape:
        raise ValueError(
            f"Shape mismatch: pred_actions {pred_actions.shape} vs gt_actions {gt_actions.shape}"
        )

    if loss_type == "l1":
        loss = F.l1_loss(pred_actions, gt_actions)
    elif loss_type == "mse":
        loss = F.mse_loss(pred_actions, gt_actions)
    elif loss_type == "smooth_l1":
        loss = F.smooth_l1_loss(pred_actions, gt_actions)
    else:
        raise ValueError(f"Unsupported loss_type: {loss_type}")

    stats = {
        "loss": loss.detach(),
        "pred_mean_abs": pred_actions.abs().mean().detach(),
        "gt_mean_abs": gt_actions.abs().mean().detach(),
        "error_mean_abs": (pred_actions - gt_actions).abs().mean().detach(),
    }

    return loss, stats

In [16]:
film = ReViPTSFiLM(
    clue_dim=clue.shape[-1],
    token_dim=patch_tokens.shape[-1],
).to(patch_tokens.device, dtype=patch_tokens.dtype)

mod_patch_tokens, aux = film(
    patch_tokens,
    clue,
    return_aux=True,
)

print(patch_tokens.shape)
print(clue.shape)
print(mod_patch_tokens.shape)
print(aux["alpha"])

torch.Size([1, 512, 4096])
torch.Size([1, 3584])
torch.Size([1, 512, 4096])
tensor(0.5000, device='cuda:0', dtype=torch.bfloat16,
       grad_fn=<SoftplusBackward0>)


In [17]:
vla_batch = make_vla_batch(
    batch=batch,
    cfg=cfg,
    vla=vla,
    processor=processor,
    proprio_projector=proprio_projector,
    append_proprio=True,   # need to be tested upon
    dtype=torch.bfloat16,
)

### Predict

In [18]:
from prismatic.vla.constants import ACTION_PROPRIO_NORMALIZATION_TYPE, NormalizationType


def append_proprio_after_revip(vla, vision_patch_embeddings, proprio, proprio_projector):
    """Append proprio after TS-FiLM, matching the original single-sample order.

    Original path: vision projection -> ReViP/TS-FiLM -> proprio projection append.
    This avoids modulating the proprio token itself.
    """
    use_proprio = proprio_projector is not None and proprio is not None

    if use_proprio:
        proprio = proprio.to(
            device=vision_patch_embeddings.device,
            dtype=vision_patch_embeddings.dtype,
        )
        projected_patch_embeddings = vla._process_proprio_features(
            vision_patch_embeddings,
            proprio,
            proprio_projector,
        )
    else:
        projected_patch_embeddings = vision_patch_embeddings

    return projected_patch_embeddings, use_proprio


def predict_normalized_actions_from_revip_batch(
    vla,
    vla_batch,
    projected_patch_embeddings,
    action_head,
):
    """Run your ReViP-aware regression method on a batched token sequence."""
    if not hasattr(vla, "_regression_prediction_revip"):
        raise AttributeError(
            "vla._regression_prediction_revip is missing. Use your patched modeling_prismatic.py "
            "that keeps normalized actions as tensors for training."
        )

    num_patches = projected_patch_embeddings.shape[1]

    return vla._regression_prediction_revip(
        vla_batch["input_embeddings"],
        vla_batch["all_actions_mask"],
        projected_patch_embeddings,
        vla_batch["attention_mask"],
        vla_batch["labels"],
        num_patches,
        vla_batch["num_prompt_tokens"],
        action_head,
    )


def unnormalize_actions_tensor(vla, normalized_actions, unnorm_key=None):
    """Torch version of action unnormalization so gradients can flow to TS-FiLM."""
    if not torch.is_tensor(normalized_actions):
        normalized_actions = torch.as_tensor(normalized_actions)

    action_norm_stats = vla.get_action_stats(unnorm_key)

    if ACTION_PROPRIO_NORMALIZATION_TYPE == NormalizationType.BOUNDS:
        ref = action_norm_stats["min"]
        mask = action_norm_stats.get("mask", np.ones_like(ref, dtype=bool))
        action_high = action_norm_stats["max"]
        action_low = action_norm_stats["min"]
    elif ACTION_PROPRIO_NORMALIZATION_TYPE == NormalizationType.BOUNDS_Q99:
        ref = action_norm_stats["q01"]
        mask = action_norm_stats.get("mask", np.ones_like(ref, dtype=bool))
        action_high = action_norm_stats["q99"]
        action_low = action_norm_stats["q01"]
    else:
        raise ValueError("Unsupported action/proprio normalization type detected!")

    mask = torch.as_tensor(
        mask,
        device=normalized_actions.device,
        dtype=torch.bool,
    )
    action_high = torch.as_tensor(
        action_high,
        device=normalized_actions.device,
        dtype=normalized_actions.dtype,
    )
    action_low = torch.as_tensor(
        action_low,
        device=normalized_actions.device,
        dtype=normalized_actions.dtype,
    )

    while mask.dim() < normalized_actions.dim():
        mask = mask.unsqueeze(0)
        action_high = action_high.unsqueeze(0)
        action_low = action_low.unsqueeze(0)

    return torch.where(
        mask,
        0.5 * (normalized_actions + 1) * (action_high - action_low + 1e-8) + action_low,
        normalized_actions,
    )

In [19]:
patch_tokens = vla_batch["projected_patch_embeddings"]
mod_patch_tokens, aux = film(patch_tokens, clue, return_aux=True)

normalized_actions, actions_hidden_states = predict_normalized_actions_from_revip_batch(
    vla,
    vla_batch,
    mod_patch_tokens,
    action_head,
)